LATER Fitting Exercise

The basic idea in fitting a model to data is to find the parameters of the model that provide in some sense the best match of the model to the data. This match is provided by the "objective function." 

This exercise is intended to demystify this process by getting you to define the initial conditions and objective function for fitting the LATER model to RT data. For a much more thorough, but still very accessible, overview of model fitting (to behavioral data), here is a great place to start:

https://elifesciences.org/articles/49547

For this exercise, recall that the point of the LATER model is that 1/RT is distributed as a Gaussian, where we can define the parameters of the Gaussian (mu and sigma) with respect to the standard parameters of the LATER model (muR and deltaS):
       mu = muR/deltaS
       sigma = 1/deltaS
       
So fitting LATER to behavioral data involves finding parameters muR and deltaS that provide the best match to the data, according to the appropriate objective function.

Follow along the steps below, some of which will require you to complete the code (and therefore hopefully think about how to relate the high-level concepts discussed above with the nitty-gritty part of getting everything to actually work.

**1. Get the data** <br>
Use this code to get a data set (array of RTs from a single condition) to fit, already preprocessed to include correct trials only and remove outliers (including express saccades). See later_getData for details <br>
    data = later_getData([], [], 0.2); <br>
    RTs = data{1}; <br>
    clear data

In [11]:
import os
import numpy as np
import scipy.io as sio

def later_getData(subjectTag='JT', dataDirectory='/Users/diane1/Documents/Classes/NGG_6050_Class/NGG6050/LATERdata', expressCutoff=0.0):
    # Load the .mat file for the specified subject
    mat_file_path = os.path.join(dataDirectory, 'data_mgl', 'F', f'{subjectTag}_RT.mat')
    mat_data = sio.loadmat(mat_file_path)

    # Extract necessary variables from the loaded data
    decisionSum = mat_data['decisionSum'].flatten()
    labelSum = mat_data['labelSum'].flatten()
    numdirSum = mat_data['numdirSum'].flatten()
    percorrSum = mat_data['percorrSum'].flatten()
    tRxnSum = mat_data['tRxnSum'].flatten()

    # Define selection criteria:
    Ltrials = (percorrSum == 1) & (tRxnSum > expressCutoff) & (tRxnSum < 1.2)

    # Data sets as defined in the comments:
    data_ = [
        tRxnSum[Ltrials & (numdirSum == -1) & (labelSum == 1)],
        tRxnSum[Ltrials & (numdirSum == -1) & (labelSum != 1)],
        tRxnSum[Ltrials & (numdirSum == 1) & (labelSum == 1)],
        tRxnSum[Ltrials & (numdirSum == 1) & (labelSum != 1)]
    ]

    # Define labels if needed
    labels_ = ['Left Choice, CP', 'Left Choice, No CP', 'Right Choice, CP', 'Right Choice, No CP']

    return data_, labels_

data= later_getData(subjectTag='JT', expressCutoff=0.2)
RTs = data[0]
print(f"RTs:{RTs}")


RTs:[array([0.486722, 0.447101, 0.461715, ..., 0.377148, 0.353642, 0.270038]), array([0.427658, 0.478042, 0.407701, 0.392792, 0.33945 , 0.576767,
       0.459844, 0.423207, 0.465359, 0.473921, 0.462475, 0.438489,
       0.478422, 0.42922 , 0.444833, 0.4805  , 0.56745 , 0.779507,
       0.449639, 0.357387, 0.375784, 0.355616, 0.376171, 0.489103,
       0.461294, 0.424784, 0.444582, 0.583888, 0.462116, 0.484841,
       0.443402, 0.536847, 0.480386, 0.423062, 0.374804, 0.394774,
       0.420718, 0.466904, 0.413918, 0.427806, 0.446184, 0.393832,
       0.71465 , 0.476448, 0.746959, 0.525793, 0.477809, 0.591425,
       0.473638, 0.449208, 0.615513, 0.40705 , 0.463594, 0.394847,
       0.391592, 0.482178, 0.36234 , 0.478844, 0.343877, 0.357569,
       0.431396, 0.358628, 0.407172, 0.39153 , 0.373235, 0.407975,
       0.357664, 0.348117, 0.32095 , 0.360003, 0.384998, 0.504198,
       0.416723, 0.453806, 0.942731, 0.739312, 0.439504, 0.403947,
       0.608911, 0.458523, 0.391084, 0.391695, 0.2

**(2) Define the objective function** <br>
The objective function typically defines the error that you want to minimize between your data and the model predictions. A common objective function is the negative of the sum of the log-likelihoods of the data, given the model parameters. To unpack that for the LATER model: <br>

    1. For each data point (RT from a single trial, in this case) and given set of model parameters, compute the probability of the data, given the model (i.e., the likelihood)

    2. Take the logarithm
    3. Sum all these log-likelihoods from all the data points
    4. Take the negative, because we want to find the minimum (thus corresponding to the maximum likelihood)

You can define the function simply using an "anonymous function"
(https://www.mathworks.com/help/matlab/matlab_prog/anonymous-functions.html), 
using this template that assumes that "fits" is a 2x1 vector of[muR, deltaS]:
 
EXERCISE:
laterErrFcn = @(fits) <**YOUR OBJECTIVE FUNCTION HERE AS A FUNCTION OF FITS**>;

**3. Define initial conditions** <br>
For the actual fitting, we will use fmincon (https://www.mathworks.com/help/optim/ug/fmincon.html), which is "function minimization with constraints." This function allows for constraints that include upper and lower bounds on the parameters. So here we define those bounds, along with the initial values. We'll use fairly arbitrary values for the lower and upper bounds, but we should pick the initial values more judiciously. 
HINT: Recall that the muR and deltaS should be strongly related to empirical summary statistics of `the (reciprocal) RT distribution.
lowerBounds = [0.001 0.001];
upperBounds = [1000 1000]; 

EXERCISE: initialValues = [<**ADD INITIAL VALUES HERE**>];

In [14]:
import os
import numpy as np
import scipy.io as sio
from scipy.stats import invgauss
from scipy.optimize import minimize

def later_likelihood(rt, mu, lambda_):
    """ Compute the likelihood of a single reaction time (RT) under the LATER model. """
    return invgauss.pdf(rt, mu / lambda_, scale=lambda_)

def log_likelihood(params, RTs):
    """ Compute the negative log-likelihood of the LATER model given the reaction times (RTs). """
    mu, lambda_ = params  # Unpack the parameters
    likelihoods = np.array([later_likelihood(rt, mu, lambda_) for rt in RTs])
    log_likelihoods = np.log(likelihoods + 1e-10)  # Adding a small value to avoid log(0)
    return -np.sum(log_likelihoods)

# Optimization function
def optimize_later_model(RTs):
    """ Optimize the LATER model parameters to minimize the negative log-likelihood. """
    # Initial guesses for mu and lambda
    initial_params = [0.5, 1.0]  # Adjust based on your knowledge of the data
   
    # Set bounds for parameters
    lowerBounds = [0.001, 0.001]
    upperBounds = [1000, 1000]
    bounds = list(zip(lowerBounds, upperBounds))  # Create a list of tuples for bounds

    # Minimize the negative log-likelihood
    result = minimize(log_likelihood, initial_params, args=(RTs,), bounds=bounds, method='L-BFGS-B')

    # Return the optimized parameters
    return result.x, result.fun

# Optimize the LATER model parameters
optimized_params, neg_log_likelihood_value = optimize_later_model(RTs)

print(f"Optimized Parameters (mu, lambda): {optimized_params}")
print(f"Negative Log-Likelihood: {neg_log_likelihood_value}")


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.

**4. Run the fits** <br>
We will be using GlobalSearch . The general advantage of this approach is to avoid local minima; for details, see:
https://www.mathworks.com/help/gads/how-globalsearch-and-multistart-work.html

_These options seem to work well, but I don't have a stronger rationale for using them. See the Matlab documentation if you really want to dive in and understand them, and let me know if you find better settings!_ <br><br>
opts = optimoptions(@fmincon,    ... % "function minimization with constraints" <br>
   'Algorithm',   'active-set',  ... <br>
   'MaxIter',     3000,          ... <br>
   'MaxFunEvals', 3000); <br><br>

_Definine the "optimization problem" using variables defined above_ <br>
problem = createOptimProblem('fmincon',    ... <br>
    'objective',   laterErrFcn,     ... # Use the objective function <br>
    'x0',          initialValues,   ... # Initial conditions <br>
    'lb',          lowerBounds,     ... # Parameter lower bounds <br>
    'ub',          upperBounds,     ... # Parameter upper bounds <br>
    'options',     opts);               # Options defined above <br><br>

_Create a GlobalSearch object_ <br>
    gs = GlobalSearch; <br>
   
Run it, returning the best-fitting parameter values and the negative-log-likelihood returned by the objective function <br>
[fits(ii,:), nllk] = run(gs,problem);

In [9]:
import numpy as np
from scipy.optimize import minimize
from scipy.optimize import Bounds

# Example implementation of the log-likelihood function (replace with your actual implementation)
def log_likelihood(RTs, mu, lambda_):
    # Placeholder for actual log-likelihood calculation
    likelihood = (1 / (lambda_ * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((RTs - mu) / lambda_)**2)
    return np.sum(np.log(likelihood))

# Define the objective function using the previously defined log_likelihood function
def laterErrFcn(params, RTs):
    mu, lambda_ = params
    return -log_likelihood(RTs, mu, lambda_)

# Initial parameter values
initialValues = [10, 1]  # Example initial guesses for mu and lambda
lowerBounds = [0.001, 0.001]  # Lower bounds for the parameters
upperBounds = [1000, 1000]  # Upper bounds for the parameters

# Define bounds for the optimization
bounds = Bounds(lowerBounds, upperBounds)

# Options for the optimization
options = {
    'maxiter': 3000,          # Maximum number of iterations
    'disp': True              # Display optimization messages
}

# Run the optimization using 'L-BFGS-B' method (similar to fmincon's active-set)
result = minimize(laterErrFcn, initialValues, args=(RTs,), method='L-BFGS-B', bounds=bounds, options=options)

# Get the optimized parameters and negative log-likelihood
optimized_params = result.x
nllk = result.fun

# Print the results
print(f"Optimized Parameters (mu, lambda): {optimized_params}")
print(f"Negative Log-Likelihood: {nllk}")

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.

**5. Evaluate the fits** <br>
EXERCISE: How do you know if you got a reasonable answer?

In [10]:
#1) Check the convergence to see if got a reasonable answer
if result.success:
    print("Optimization converged successfully.")
else:
    print("Optimization did not converge:", result.message)

#2) lower negative log-likelihoods indicate a better/good fit to the data

#3) Visualize the fit
import matplotlib.pyplot as plt

# Generate predicted RTs based on optimized parameters
mu_opt, lambda_opt = optimized_params
predicted_RTs = mu_opt + lambda_opt * np.random.randn(len(RTs))  # Example prediction generation

plt.scatter(RTs, predicted_RTs, label='Fitted Values')
plt.scatter(RTs, RTs, label='Observed Data', alpha=0.5)
plt.xlabel('Observed RTs')
plt.ylabel('Fitted RTs')
plt.title('Observed vs Fitted Reaction Times')
plt.legend()
plt.show()

def compute_aic(nll, num_params):
    return 2 * nll + 2 * num_params

#4) Evaluate fit statistics-- Consider computing fit statistics like the Akaike Information Criterion (AIC) or Bayesian Information Criterion (BIC), which penalize model complexity. 
## Lower AIC/BIC values indicate a better fit.
aic = compute_aic(nllk, len(optimized_params))
print(f"AIC: {aic}")


NameError: name 'result' is not defined